# Data Preprocessing Practice

**Topic:** Data Processing and Features &nbsp;·&nbsp; **Difficulty:** Medium

## The data (created in the next cell)

- `train_df` (3,000 rows) and `test_df` (1,000 rows), loan applications with:
  - `income_k`: numeric, ~10% missing (`NaN`)
  - `debt_ratio`: stored as **strings**, with `"?"` for missing
  - `age`, `credit_lines`: clean numeric columns
  - `region`: categorical, but written inconsistently (`"west"`, `"West"`, `"WEST "` …)
  - `applicant_id`: a meaningless row ID
- `train_df["approved"]` is the 0/1 label. `y_test` is used by the evaluation cell.

## Your task

1. Clean both DataFrames the same way: coerce `debt_ratio` to numeric, impute missing values (median is fine), normalize the `region` strings and encode them, and drop `applicant_id`.
2. Train any classifier you like on the cleaned training data.
3. Set `predictions` to your 0/1 predictions for `test_df`.

## Evaluation

Test accuracy must reach the target shown in the evaluation cell.


In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(29)

def make(n, seed):
    r = np.random.default_rng(seed)
    income = r.gamma(4.0, 18.0, n)                      # in $k
    debt = r.uniform(0.02, 0.85, n)
    age = r.integers(21, 70, n)
    lines = r.poisson(4.0, n)
    region = r.choice(4, n)                             # 0..3
    logit = 0.09*income - 10.4*debt + 0.04*age + 0.30*lines + 1.0*(region == 2) - 4.4
    y = (1 / (1 + np.exp(-logit)) > r.uniform(0, 1, n)).astype(int)
    return income, debt, age, lines, region, y

def messy_frame(income, debt, age, lines, region, y, seed, with_label):
    r = np.random.default_rng(seed)
    n = len(y)
    region_names = np.array(["north", "south", "west", "east"])[region]
    styles = [str.lower, str.upper, str.title, lambda s: s + " "]
    region_messy = [styles[r.integers(4)](s) for s in region_names]
    income_col = income.round(1).astype(object)
    income_col[r.random(n) < 0.10] = np.nan
    debt_col = np.array([f"{v:.3f}" for v in debt], dtype=object)
    debt_col[r.random(n) < 0.08] = "?"
    df = pd.DataFrame({
        "applicant_id": [f"AP{seed}{i:05d}" for i in range(n)],
        "income_k": income_col.astype(float),
        "debt_ratio": debt_col,
        "age": age,
        "credit_lines": lines,
        "region": region_messy,
    })
    if with_label:
        df["approved"] = y
    return df

_tr = make(3000, 61)
_te = make(1000, 62)
train_df = messy_frame(*_tr, seed=71, with_label=True)
test_df = messy_frame(*_te, seed=72, with_label=False)
y_test = _te[5]

print(train_df.dtypes)
train_df.head()


In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.





In [ ]:
# =====================================================================
# EVALUATION: run this cell to compare your predictions to the truth
# =====================================================================
import numpy as np

TARGET_ACCURACY = 0.78

if "predictions" not in dir():
    raise NameError("Define `predictions` in the solution cell before running this cell.")

_pred = np.asarray(predictions).ravel()
_true = np.asarray(y_test).ravel()
if _pred.shape != _true.shape:
    raise ValueError(f"`predictions` has shape {_pred.shape}, expected {_true.shape}.")

accuracy = float((_pred == _true).mean())
print("=" * 52)
print(f"Test accuracy : {accuracy:.4f}")
print(f"Target        : >= {TARGET_ACCURACY:.2f}")
print("-" * 52)
print("Per-class recall:")
for _cls in np.unique(_true):
    _mask = _true == _cls
    print(f"  class {_cls!r:>6} : {float((_pred[_mask] == _cls).mean()):.3f}   ({int(_mask.sum())} test samples)")
print("=" * 52)
if accuracy >= TARGET_ACCURACY:
    print("PASSED: nice work!")
else:
    print("NOT PASSED yet: revisit your model and try again.")


## Solution

Clean train and test with the same function: coerce `debt_ratio` with `pd.to_numeric(errors='coerce')`, impute medians taken from the *training* table, normalize the region strings before one-hot encoding, and drop the ID column. A logistic regression on the cleaned features clears the bar.

```python
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

def clean(df, medians=None):
    out = df.drop(columns=["applicant_id"]).copy()
    out["debt_ratio"] = pd.to_numeric(out["debt_ratio"], errors="coerce")
    out["region"] = out["region"].str.strip().str.lower()
    if medians is None:
        medians = out[["income_k", "debt_ratio"]].median()
    out[["income_k", "debt_ratio"]] = out[["income_k", "debt_ratio"]].fillna(medians)
    out = pd.get_dummies(out, columns=["region"])
    return out, medians

train_clean, medians = clean(train_df.drop(columns=["approved"]))
test_clean, _ = clean(test_df, medians)
test_clean = test_clean.reindex(columns=train_clean.columns, fill_value=0)

clf = LogisticRegression(max_iter=3000)
clf.fit(train_clean, train_df["approved"])
predictions = clf.predict(test_clean)
```
